In [1]:
import itertools

def get_cost(shape_a, shape_b, shared_indices):
    """Вычисляет стоимость сокращения двух тензоров и размерность результата."""
    cost = 1
    # Стоимость = произведение всех уникальных индексов в обоих тензорах
    combined_indices = set(shape_a.keys()) | set(shape_b.keys())
    for idx in combined_indices:
        cost *= shape_a.get(idx, shape_b.get(idx))

    # Результирующая форма (индексы, которые не сократились)
    new_shape = {k: v for k, v in {**shape_a, **shape_b}.items()
                 if k not in (shape_a.keys() & shape_b.keys())}
    return cost, new_shape

def optimize_einsum(equation, shapes):
    """DP алгоритм поиска оптимального пути."""
    inputs = equation.split('->')[0].split(',')
    # Превращаем в список словарей {индекс: размер}
    tensor_shapes = [{idx: shapes[idx] for idx in term} for term in inputs]
    n = len(tensor_shapes)

    # dp[mask] = (min_cost, resulting_shape, strategy_tree)
    dp = {}

    # Базовые случаи: одиночные тензоры
    for i in range(n):
        dp[1 << i] = (0, tensor_shapes[i], i)

    # Перебор размеров подмножеств
    for length in range(2, n + 1):
        for subset in itertools.combinations(range(n), length):
            mask = sum(1 << i for i in subset)
            best_cost = float('inf')

            # Разделяем маску на две части (L и R)
            for i in range(1, (1 << length) - 1):
                # Генерируем подмаску
                submask_l = 0
                bit_idx = 0
                for j in range(n):
                    if (mask >> j) & 1:
                        if (i >> bit_idx) & 1:
                            submask_l |= (1 << j)
                        bit_idx += 1

                submask_r = mask ^ submask_l
                if submask_l > submask_r: continue # Избегаем дублей

                if submask_l in dp and submask_r in dp:
                    cost_l, shape_l, tree_l = dp[submask_l]
                    cost_r, shape_r, tree_r = dp[submask_r]

                    # Находим общие индексы для сокращения
                    shared = set(shape_l.keys()) & set(shape_r.keys())
                    current_cost, new_shape = get_cost(shape_l, shape_r, shared)
                    total_cost = cost_l + cost_r + current_cost

                    if total_cost < best_cost:
                        best_cost = total_cost
                        dp[mask] = (best_cost, new_shape, (tree_l, tree_r))

    return dp[(1 << n) - 1]

In [5]:
dims = {'i': 10, 'j': 100, 'k': 10, 'l': 100, 'm': 10}
equation = "ij,jk,kl,lm->im"

best_cost, final_shape, path = optimize_einsum(equation, dims)

print(f"Минимальная стоимость: {best_cost}")
print(f"Дерево сокращений: {path}")

Минимальная стоимость: 21000
Дерево сокращений: ((0, 1), (2, 3))
